## Monte Carlo integration: mathematical idea

We want to estimate the definite integral

$$
I=\int_a^b f(x)\,dx.
$$

Monte Carlo integration is based on **random sampling**.  
Suppose we choose random points $x_1,x_2,\dots,x_N$ uniformly from the interval $[a,b]$.  
Then the average value of the function over these random points,

$$
\frac{1}{N}\sum_{i=1}^N f(x_i),
$$

approximates the true average value of the function on $[a,b]$, which is

$$
\frac{1}{b-a}\int_a^b f(x)\,dx.
$$

Rearranging gives the Monte Carlo estimate of the integral:

$$
\int_a^b f(x)\,dx \approx (b-a)\,\frac{1}{N}\sum_{i=1}^N f(x_i).
$$

So the method works in three mathematical steps:

1. Sample $N$ random points uniformly in $[a,b]$
2. Compute the average value of $f(x)$ at those points
3. Multiply by the interval width $(b-a)$

### Example

For

$$
f(x)=x^2, \qquad a=0,\quad b=1,
$$

the exact integral is

$$
\int_0^1 x^2\,dx=\left[\frac{x^3}{3}\right]_0^1=\frac{1}{3}.
$$

Using Monte Carlo integration, we estimate this as

$$
\int_0^1 x^2\,dx \approx \frac{1}{N}\sum_{i=1}^N x_i^2,
$$

since here $(b-a)=1$.

As $N$ becomes large, this estimate approaches the exact value $1/3$.

### Why it works

This method follows from the **law of large numbers**: the sample mean of many random values approaches the expected value.  
If $X$ is uniformly distributed on $[a,b]$, then

$$
\mathbb{E}[f(X)] = \frac{1}{b-a}\int_a^b f(x)\,dx.
$$

Therefore,

$$
\int_a^b f(x)\,dx = (b-a)\,\mathbb{E}[f(X)],
$$

and Monte Carlo integration replaces the expectation by a sample average.

### Accuracy

The estimate is not exact because it depends on random samples.  
Its error typically decreases like

$$
\text{error} \sim \frac{1}{\sqrt{N}},
$$

so using more samples improves the estimate, although convergence is relatively slow.

In [1]:
import numpy as np

def monte_carlo_integration(f, a, b, n_samples=100000):
    # Generate random samples in [a, b]
    x = np.random.uniform(a, b, n_samples)
    
    # Evaluate function
    y = f(x)
    
    # Monte Carlo estimate
    integral = (b - a) * np.mean(y)
    
    return integral


# Example: ∫₀¹ x² dx = 1/3
f = lambda x: x**2

result = monte_carlo_integration(f, 0, 1, n_samples=1000000)
print("Estimated integral:", result)
print("Exact value:", 1/3)

Estimated integral: 0.33301044176168415
Exact value: 0.3333333333333333


## Monte Carlo simulation of electron transport

This simulation models the motion of electrons in a semiconductor under an applied electric field. The key physical idea is that electrons are accelerated by the field between collisions, while scattering events randomize their motion. The competition between these two processes produces a finite **drift velocity**.

### Electron acceleration in an electric field

An electron in an electric field $E$ experiences a force

$$
F = qE,
$$

so its acceleration is

$$
a = \frac{qE}{m_{\mathrm{eff}}},
$$

where $q$ is the electron charge and $m_{\mathrm{eff}}$ is the effective mass in the semiconductor.

If there were no scattering, the electron velocity would increase continuously with time according to

$$
v(t+\Delta t) = v(t) + a\,\Delta t.
$$

### Role of scattering

In a real material, electrons do not accelerate forever. They undergo random scattering events due to phonons, impurities, and other interactions. These collisions interrupt the motion and randomize the electron velocity.

If the mean free time between collisions is $\tau$, then the probability that an electron scatters during a small time interval $\Delta t$ is

$$
P_{\mathrm{scatter}} = 1 - e^{-\Delta t/\tau}.
$$

For very small $\Delta t$, this becomes approximately

$$
P_{\mathrm{scatter}} \approx \frac{\Delta t}{\tau}.
$$

Thus, during each time step, each electron has a small random chance of scattering.

### Thermalization after scattering

When a scattering event occurs, the electron velocity is reset to a random value drawn from a thermal distribution. This represents the idea that collisions destroy the memory of the previous directed motion and restore a more random thermal motion.

So the motion of each electron consists of:

- acceleration by the electric field between collisions,
- random velocity reset when scattering occurs.

The velocity reset uses a Gaussian width of about $10^5\ \mathrm{m/s}$, which is a reasonable thermal velocity scale at room temperature.

Using equipartition for one velocity component,

$$
\frac{1}{2}m_{\mathrm{eff}}\langle v_x^2\rangle = \frac{1}{2}k_B T,
$$

so the 1D thermal velocity scale is

$$
v_{\mathrm{th}} \sim \sqrt{\frac{k_B T}{m_{\mathrm{eff}}}}.
$$

For $T=300\ \mathrm{K}$ and

$$
m_{\mathrm{eff}} = 0.26\,m_e = 0.26(9.11\times10^{-31})\ \mathrm{kg}
\approx 2.37\times10^{-31}\ \mathrm{kg},
$$

we get

$$
v_{\mathrm{th}}
=
\sqrt{\frac{(1.38\times10^{-23})(300)}{2.37\times10^{-31}}}
\approx 1.3\times10^5\ \mathrm{m/s}.
$$

Thus the chosen standard deviation $10^5\ \mathrm{m/s}$ is of the correct order of magnitude for thermally randomized electron velocities.

### Ensemble drift velocity

Although each individual electron moves randomly, the average motion of many electrons becomes predictable. The average velocity of the electron ensemble is called the **drift velocity**:

$$
v_d(t) = \frac{1}{N}\sum_{i=1}^N v_i(t),
$$

where $N$ is the number of simulated electrons.

Initially, the drift velocity increases because the electric field accelerates the electrons. After some time, scattering balances the acceleration, and the drift velocity approaches a steady average value.


### Physical interpretation

This is a simple Monte Carlo version of the Drude transport picture:

- the electric field drives the electrons forward,
- scattering randomizes their motion,
- the balance of the two gives a finite average drift.

A useful approximate expression for the steady-state drift velocity is

$$
v_d \approx \frac{qE\tau}{m_{\mathrm{eff}}}.
$$

This shows that the drift velocity increases with:

- stronger electric field $E$,
- longer mean free time $\tau$,

and decreases with:

- larger effective mass $m_{\mathrm{eff}}$.

### Why this is called Monte Carlo

The simulation uses **random numbers** to decide when scattering happens. The trajectory of any one electron is unpredictable, but the average over many electrons reproduces the expected transport behavior. This is the essence of the Monte Carlo method: using random sampling to model a physical process with known statistical rules.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -----------------------------
# Physical constants
# -----------------------------
q = 1.6e-19          # Electron charge (C)
m_eff = 0.26 * 9.11e-31  # Effective mass (kg, Silicon)
tau = 0.2e-11        # Mean free time (s)
E_field = 1e5        # Electric field (V/m)

# -----------------------------
# Simulation parameters
# -----------------------------
dt = 1e-15           # Time step (s)
steps = 20000        # Number of time steps
num_electrons = 1000

# -----------------------------
# Initialization
# -----------------------------
velocities = np.zeros(num_electrons)
positions = np.zeros(num_electrons)

drift_velocity = []

# Precompute acceleration
a = q * E_field / m_eff

# -----------------------------
# Monte Carlo loop
# -----------------------------
for step in range(steps):
    # Acceleration update
    velocities += a * dt

    # Scattering probability
    P_scatter = 1 - np.exp(-dt / tau)

    # Random scattering events
    random_numbers = np.random.rand(num_electrons)
    scatter_indices = random_numbers < P_scatter

    # Reset velocity after scattering (thermalized)
    velocities[scatter_indices] = np.random.normal(0, 1e5, np.sum(scatter_indices))

    # Update positions
    positions += velocities * dt

    # Record average drift velocity
    drift_velocity.append(np.mean(velocities))

# -----------------------------
# Plot results
# -----------------------------
time = np.arange(steps) * dt

plt.figure(figsize=(8,5))
plt.plot(time * 1e12, drift_velocity)
plt.xlabel("Time (ps)")
plt.ylabel("Drift Velocity (m/s)")
plt.title("Monte Carlo Simulation of Electron Transport")
plt.grid()
plt.show()